# R-tag pipeline — single run walkthrough

This notebook runs the **Python R-tag (RTEG) workflow** for the KB331 sample filter.

**Manual reference:** Jing Yang's SKILL flow in [`../rdsBawTEGAutoFromTemp.il`](../rdsBawTEGAutoFromTemp.il)  
**Documentation:** [`../README.md`](../README.md) · [`../CLAUDE.md`](../CLAUDE.md)

Run all cells top-to-bottom from the `python_code/` directory.

| Step | Section | Source file(s) | Primary functions |
|------|---------|----------------|-------------------|
| 1 | Setup / inputs | `layermap.py` | `load_layermap` |
| 2.1 | Layermap | `layermap.py` | `load_layermap` |
| 2.2 | Inspect refs | `inspect_refs.py` | `inspect_gds` |
| 2.3 | Identify | `separate.py` | `identify` |
| 3 | PPD + orientation placement | `prep_resonator_ppd.py`, `rteg_orientation.py`, `rteg_collect.py` | `prep_resonator_ppd`, `analyze_orientation`, `preserved_collars_at_shift` |
| 4 | Die frame | `prep_rteg_frame.py`, `export_gds.py` | `prep_rteg_in_frame`, `export_gds` |
| 5.1 | Collect roles | `rteg_collect.py` | `collect_geometry_roles` |
| 5.2 | Classify / orientation | `rteg_orientation.py`, `rteg_classify.py` | `collect_orientation_inputs`, `classify_nodes` |
| 5.3 | MTE extensions | `rteg_mte_extensions.py` | `build_mte_extensions` → collar → lip A/B → draw → `is_connected` |
| 5.3 export | MTE GDS | `rteg_mte_extensions.py`, `export_gds.py` | `export_mte_extensions_gds` |
| 5.4 | Route MTE (future) | `rteg_mte_route.py` | `build_mte_pad_routes` |


In [1]:
from __future__ import annotations
import io
import sys
from contextlib import redirect_stdout
from pathlib import Path
import gdstk
import pandas as pd


def resolve_python_code_root() -> Path:
    """Find python_code/ by looking for input_files/ + src/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "input_files").is_dir() and (candidate / "src").is_dir():
            return candidate
    return here


ROOT = resolve_python_code_root()
SRC = ROOT / "src"
INPUT_FILES = ROOT / "input_files"
DRAFT = ROOT / "draft_output"
ORIGNAL_RTEG = DRAFT / "original_rteg"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DRAFT.mkdir(parents=True, exist_ok=True)

print(f"Working directory: {ROOT}")
print(f"Input files:       {INPUT_FILES}")
print(f"Draft output:      {DRAFT}")
print(f"Source code:       {SRC}")

Working directory: C:\Users\santosr4\Documents\rTEG Automation\python_code
Input files:       C:\Users\santosr4\Documents\rTEG Automation\python_code\input_files
Draft output:      C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output
Source code:       C:\Users\santosr4\Documents\rTEG Automation\python_code\src


## Define Inputs Here For Running This Notebook

In [2]:
FILTER = INPUT_FILES / "KB331_N_01.gds" # input filter GDS file
FRAME = INPUT_FILES / "KB331_N_Frame.gds" # die frame
PPD = INPUT_FILES / "GSG_frame.gds" # GSG ppd frame
LAYERMAP = INPUT_FILES / "layermap" # Skyworks layer map

## 1. Process Inputs

In [3]:
# Ensure all referenced input files exist; abort on missing inputs

input_files = {
    "Filter layout": FILTER,
    "Frame template": FRAME,
    "Probe device": PPD,
    "Layer table": LAYERMAP,
}

input_roles = {
    "Filter layout": "Clean hierarchical filter GDS — resonators + connect metal",
    "Frame template": None,
    "Probe device": "ppd_1port — GSG pad reference",
    "Layer table": "Skyworks name → GDS (layer, datatype)",
}

rows = []
frame_size_str = "unknown size"

for label, path in input_files.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing required input: {label} ({path})")
    size = f"{path.stat().st_size:,} B"
    rows.append({"file": label, "path": path.name, "exists": True, "size": size, "role": input_roles[label]})

frame_lib = gdstk.read_gds(FRAME)
frame_cell = frame_lib.top_level()[0]
frame_bb = frame_cell.bounding_box()
if frame_bb is not None:
    (fx0, fy0), (fx1, fy1) = frame_bb
    frame_size_str = f"{fx1 - fx0:.1f}×{fy1 - fy0:.1f} µm"

for row in rows:
    if row["file"] == "Frame template":
        row["role"] = f"{frame_size_str} GSG probe frame (six BAW_MB2 pads)"

display(pd.DataFrame(rows))


,file,path,exists,size,role
0,Filter layout,KB331_N_01.gds,True,"655,360 B",Clean hierarchical filter GDS — resonators + c...
1,Frame template,KB331_N_Frame.gds,True,"34,816 B",460.0×580.0 µm GSG probe frame (six BAW_MB2 pads)
2,Probe device,GSG_frame.gds,True,"10,240 B",ppd_1port — GSG pad reference
3,Layer table,layermap,True,"3,971 B","Skyworks name → GDS (layer, datatype)"


## 2. Selection

2.1 start with layermap definitions

2.2 inspect layer references

2.3 start identifying resonators from filter GDS file

### 2.1 `layermap.py` — layer name lookups

Loads the Skyworks layermap file from `LAYERMAP` (defined above). Maps names like `BAW_MBE` to GDS `(layer, datatype)` pairs.

In [4]:
from src.layermap import load_layermap

layermap = load_layermap(LAYERMAP)
print(f"Loaded {len(layermap)} layers from {LAYERMAP.name}")

for name in ("BAW_MBE", "BAW_MTE", "BAW_MB2", "BAW_EDGE"):
    layer, dt = layermap.pair(name)
    print(f"  {name:12s} -> GDS ({layer}, {dt})")

Loaded 155 layers from layermap
  BAW_MBE      -> GDS (2, 0)
  BAW_MTE      -> GDS (5, 0)
  BAW_MB2      -> GDS (12, 0)
  BAW_EDGE     -> GDS (9, 0)


### 2.2 `inspect_refs.py` — hierarchy sanity check

Lists placed references in the filter GDS: resonators, vias, connect cells. Useful when instance names did not survive export.

In [5]:
from src.inspect_refs import inspect_gds

buf = io.StringIO()
with redirect_stdout(buf):
    inspect_gds(FILTER)

# 
print(buf.getvalue()[:2000])
if len(buf.getvalue()) > 2000:
    print("... (truncated)")


=== KB331_N_01.gds ===

shuntq3_CDNS_781040784740: 80 polys, bbox (-127.4, -49.5)-(128.1, 54.5) (no references)
   labels (9):
     'P1' @ (0.0, -0.0)  layer=(5,0)
     'P2' @ (0.0, -0.0)  layer=(2,0)
     'freq=1478M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 9.5)  layer=(221,0)
     '[@instanceName]' @ (0.0, 36.0)  layer=(221,0)
     'ReW=-99' @ (0.0, 1.5)  layer=(221,0)
     'MRaW=2.2' @ (0.0, -6.5)  layer=(221,0)
     'MTE_CLEN=172.6' @ (0.0, -14.5)  layer=(221,0)
     'MBE_CLEN=377.9' @ (0.0, -22.5)  layer=(221,0)

seriesq3_CDNS_781040784741: 93 polys, bbox (-37.6, -53.2)-(43.3, 53.2) (no references)
   labels (9):
     'P1' @ (0.0, 0.0)  layer=(5,0)
     'P2' @ (0.0, 0.0)  layer=(2,0)
     'freq=1541M INFRAver=3.0 ModelID=430 Band=nil multiKt2=None pcType=Q' @ (0.0, 0.0)  layer=(100,0)
     'ORaW=3.4' @ (0.0, 5.0)  layer=(221,0)
     '[@instanceName]' @ (0.0, 10.0)  layer=(221,0)
     'ReW=1.8' @ (0.0, 0.0)  laye

### 2.3 `separate.py` — resonator and vtb identification

Finds placed resonators (masters: `series`, `shunt`, `rcap`, `mimcap`) and `vtb` vias under the filter parent cell. Returns dataframe-ready rows via `identify(FILTER)`.

In [6]:
from src.separate import identify

identification = identify(FILTER)

parent = identification.parent
filter_lib = identification.library
res_list = identification.resonators
vias = identification.vias

res_df = pd.DataFrame(identification.resonator_rows()) #DF of resonators 
via_df = pd.DataFrame(identification.via_rows()) #DF of VIAs

print(f"Parent cell: {parent}")
print(f"Resonators: {len(res_list)}  |  Vias: {len(vias)}")

res_df

Parent cell: KB331_N_01
Resonators: 8  |  Vias: 4


,index,inst_name,master_name,type,origin_x,origin_y,rotation_deg,split_base
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,282.6,183.1,0.0,None
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,234.0,98.3,180.0,None
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,95.8,145.1,90.0,None
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,92.0,217.4,270.0,None
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,311.9,458.9,90.0,None
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,157.8,412.7,0.0,None
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,307.6,317.6,270.0,None
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,187.7,294.1,270.0,None


In [7]:
via_df

,index,master_name,origin_x,origin_y,rotation_deg
0,0,vtb3_CDNS_781040784743,208.6,127.4,270.0
1,1,vtb3_CDNS_781040784746,335.0,158.9,270.0
2,2,vtb3_CDNS_781040784749,122.6,176.9,270.0
3,3,vtb3_CDNS_781040784749,65.2,185.8,90.0


## 3. Separation
For each resonator found place it together with the GSG ppd frame in the center

### 3 — PPD + orientation placement

**Files:** `src/prep_resonator_ppd.py`, `src/rteg_orientation.py`, `src/rteg_collect.py`

**Entry points:** `prep_resonator_ppd` (with `identification` + `layermap`) → `preserved_collars_at_shift` → `analyze_orientation` → extra `orientation_shift` on placement.

For each row in `res_df`, combine the resonator with the GSG PPD frame (`GSG_frame.gds`):
center on the template, nudge for pad / release-hole clearance, then apply collar-orientation shift.


In [8]:
from IPython.display import HTML, display

from src.prep_resonator_ppd import (
    MIN_RELEASE_HOLE_CLEARANCE_UM,
    assemblies_summary_df,
    prep_resonator_ppd,
)

ppd_assemblies = prep_resonator_ppd(
    res_df,
    res_list,
    PPD,
    identification=identification,
    layermap=layermap,
)
display(assemblies_summary_df(ppd_assemblies))


,index,inst_name,master_name,type,ppd_origin_x,ppd_origin_y,resonator_origin_x,resonator_origin_y,centering_shift_x,centering_shift_y,clearance_shift_x,clearance_shift_y,orientation_shift_x,orientation_shift_y,min_release_clearance_um,shift_x,shift_y,assembly_center_x,assembly_center_y
0,0,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,498.6,222.8,105.2,59.7,90.8,0.0,20.0,-20.0,37.2,216.0,39.7,388.2,245.3
1,1,shuntq3_CDNS_781040784740,shuntq3_CDNS_781040784740,shunt,0.0,0.0,500.3,222.8,153.8,144.5,92.5,0.0,20.0,-20.0,34.1,266.3,124.5,388.2,245.3
2,2,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,414.4,245.3,289.5,100.2,19.0,0.0,10.0,0.0,32.8,318.6,100.2,388.2,245.3
3,3,seriesq3_CDNS_781040784741,seriesq3_CDNS_781040784741,series,0.0,0.0,412.4,245.3,293.3,27.9,17.0,0.0,10.0,0.0,31.5,320.4,27.9,388.2,245.3
4,4,shuntq3_CDNS_781040784742,shuntq3_CDNS_781040784742,shunt,0.0,0.0,468.7,245.8,89.0,-213.1,47.8,0.0,20.0,0.0,32.9,156.8,-213.1,388.2,245.3
5,5,shuntq3_CDNS_781040784745,shuntq3_CDNS_781040784745,shunt,0.0,0.0,475.8,221.8,230.1,-180.9,77.9,0.0,10.0,-10.0,30.8,318.0,-190.9,388.2,245.3
6,6,seriesq3_CDNS_781040784747,seriesq3_CDNS_781040784747,series,0.0,0.0,423.7,225.3,78.8,-72.3,17.3,0.0,20.0,-20.0,37.8,116.1,-92.3,388.2,245.3
7,7,seriesq3_CDNS_781040784748,seriesq3_CDNS_781040784748,series,0.0,0.0,448.0,235.6,193.4,-48.5,56.9,0.0,10.0,-10.0,31.8,260.3,-58.5,388.2,245.3


simply preview of the resonator + ppd GSG placement within the notebook directly 

In [9]:
# # Display/Preview of frame within this notebook

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="ppd-preview-item">'
#     f'<div class="ppd-preview-label">[{asm.index}] {asm.inst_name} ({asm.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in ppd_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .ppd-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .ppd-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .ppd-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .ppd-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="ppd-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

## 4. Setting Up

place the ppd+resonator combo now into the original die frame.

for the width (x axis) leave 4% margin in both sides

for the height (y axis) leave 7% margin in both sides

### `prep_rteg_frame.py` — die frame placement

For each `ppd_assembly`, place the step-2 `top_cell` inside `FRAME`:

1. Die frame at `(0, 0)`; margins measured from the **inner** MBE die frame (not the outer 460×580 bbox)
2. Assembly left edge at 3.5% inside the inner frame; Y centered with 7% margins
3. MBE filler rectangle on the right: inner-frame center → margined right edge, same height as the placed GSG/resonator assembly

Returns `rteg_assemblies` and `rteg_files_df`. Step 5 (later) adds interconnect routing and DRC.

In [10]:
from src.prep_rteg_frame import (
    assemblies_summary_df,
    prep_rteg_in_frame,
    preview_assembly_svg,
)

rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
rteg_files_df = assemblies_summary_df(rteg_assemblies)

print(f"Built {len(rteg_assemblies)} RTEG frame assemblies")
display(rteg_files_df)

Built 8 RTEG frame assemblies


C:\Users\santosr4\AppData\Local\Temp\ipykernel_24016\573027339.py:7: UserWarning: Assembly [0] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)
C:\Users\santosr4\AppData\Local\Temp\ipykernel_24016\573027339.py:7: UserWarning: Assembly [1] shuntq3_CDNS_781040784740 extends past the 4.0%/7.0% content box inside the inner die frame
  rteg_assemblies = prep_rteg_in_frame(ppd_assemblies, FRAME)


,index,inst_name,type,frame_origin_x,frame_origin_y,assembly_origin_x,assembly_origin_y,content_center_x,content_center_y,inner_frame_x0,...,inner_frame_x1,inner_frame_y1,mbe_filler_x0,mbe_filler_y0,mbe_filler_x1,mbe_filler_y1,inner_content_width,mbe_filler_width,x_margin_pct,y_margin_pct
0,0,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
1,1,shuntq3_CDNS_781040784740,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
2,2,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
3,3,seriesq3_CDNS_781040784741,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
4,4,shuntq3_CDNS_781040784742,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
5,5,shuntq3_CDNS_781040784745,shunt,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
6,6,seriesq3_CDNS_781040784747,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07
7,7,seriesq3_CDNS_781040784748,series,0.0,0.0,-216.2,44.7,230.0,290.0,36.5,...,423.5,543.5,230.0,102.5,408.0,477.5,356.0,178.0,0.04,0.07


In [11]:
# from IPython.display import HTML, display

# _preview_cols = 4
# _preview_items = "\n".join(
#     f'<div class="rteg-preview-item">'
#     f'<div class="rteg-preview-label">[{asm.index}] {asm.inst_name} ({asm.ppd_assembly.res_type})</div>'
#     f'{preview_assembly_svg(asm)}'
#     f"</div>"
#     for asm in rteg_assemblies
# )
# display(
#     HTML(
#         f"""
# <style>
# .rteg-preview-grid {{
#   display: grid;
#   grid-template-columns: repeat({_preview_cols}, minmax(0, 1fr));
#   gap: 8px;
#   margin-top: 8px;
# }}
# .rteg-preview-item {{
#   border: 1px solid #ccc;
#   padding: 4px;
#   text-align: center;
#   overflow: hidden;
# }}
# .rteg-preview-label {{
#   font-size: 11px;
#   margin-bottom: 4px;
# }}
# .rteg-preview-item svg {{
#   width: 100%;
#   height: auto;
#   display: block;
# }}
# </style>
# <div class="rteg-preview-grid">
# {_preview_items}
# </div>
# """
#     )
# )

In [12]:
from src.export_gds import export_gds, export_summary_df

rteg_export_df = export_summary_df(
    export_gds(
        rteg_assemblies,
        ORIGNAL_RTEG,
        layermap=layermap,
        parent=parent,
        stage="framed",
    )
)

print(f"Exported {len(rteg_export_df)} GDS files to {ORIGNAL_RTEG}")
print("Layer names: open each .gds with its matching .lyp in KLayout")
display(rteg_export_df)


Exported 8 GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\original_rteg
Layer names: open each .gds with its matching .lyp in KLayout


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_0_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117740
1,1,shuntq3_CDNS_781040784740,rteg_1_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117758
2,2,seriesq3_CDNS_781040784741,rteg_2_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124608
3,3,seriesq3_CDNS_781040784741,rteg_3_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124608
4,4,shuntq3_CDNS_781040784742,rteg_4_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,136288
5,5,shuntq3_CDNS_781040784745,rteg_5_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,134694
6,6,seriesq3_CDNS_781040784747,rteg_6_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,122800
7,7,seriesq3_CDNS_781040784748,rteg_7_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,120968


========================================================

## 5. Routing — solving interconnect algorithm

**Goal:** turn a framed resonator into a DRC-clean RTEG — one fused MBE ground body with two pockets carved out (resonator + signal).

### Pipeline overview

| Step | What it does | Primary call |
|------|----------------|--------------|
| **5.1 Collect** | Split the framed layout into typed polygon sets (ground plates, preserved filter metal, release holes, frame boundary, body MTE). | `collect_geometry_roles` |
| **5.2 Classify** | Label GSG bands signal vs ground from collar orientation; decide later pad-routing mode. | `classify_nodes` |
| **5.3 MTE extensions** | For **every** resonator with preserved MTE, draw one outward collar extension (~14 µm). | `build_mte_extensions` |
| **5.4 Route MTE** | (Future) Connect center-facing extensions to the signal pad. | `build_mte_pad_routes` |
| **5.5–5.7** | Union ground, carve pockets, reconnect preserved metal. | TBD |

### MTE extension algorithm (5.3) — at a glance

Step 5.3 does **not** use 5.2 classification yet. It runs the same deterministic geometry pipeline on each resonator:

1. **Inputs** — `RtegGeometryRoles` from 5.1 (`preserved.mte`, `resonator_body_mte`).
2. **Collar** — `select_extension_collar` picks the small mouth polygon the extension will merge into (never a disjoint die-wide bus).
3. **Mouth A / B** — `find_outward_lip_ab` scans the collar perimeter, scores outward-facing edges, and sets intercept corners **A** and **B** on the winning lip edge.
4. **Draw** — `draw_lip_extension` builds a rectangle: inward merge into the collar at A/B, then 14 µm outward.
5. **DONE** — `extension_is_connected` must pass; `MteExtensionResult.is_connected == True` means the extension is materially merged, not just touching at a sliver.

See **§ 5.3** below for the full step-by-step breakdown and all tunable config fields.

### Remaining routing steps (after MTE)

5. **Union ground (5.5)** — OR ground node blocks + filler + preserved MBE.
6. **Carve pockets (5.6)** — subtract signal net, resonator keep-out, release holes.
7. **Reconnect (5.7)** — union preserved ground metal back; drop slivers.


### 5.1 — Collect geometry roles

**Files:** `src/rteg_collect.py`

**Entry point:** `collect_geometry_roles(frame_assembly, res_info, identification, layermap, config?)`

**What this cell does:** runs collection once per resonator and stores the result in `all_roles[index]`. Step 5.3 reads `all_roles` — nothing is drawn here.

#### Algorithm

For each framed resonator, `collect_geometry_roles` builds an `RtegGeometryRoles` bundle:

1. **Ground plates** — GSG probe-side MBE polygons (top / center / bottom / filler) from the frame template.
2. **Preserved MBE / MTE** — filter interconnect metal copied from the parent die (NPI mandate: exact replication near the resonator).
3. **Release holes** — `BAW_ReF` / `BAW_CAV` outlines near the resonator body.
4. **Frame boundary** — inner routable cavity + frame MBE ring.
5. **Resonator body MTE** — inner resonator metal on `BAW_MTE` (used later to score collar overlap and outward direction).

#### Preserved MTE collection (`connectMTE`, layermap 5/0)

This is the input set that step 5.3 extends from:

1. Collect all `connectMTE` polygons whose bbox overlaps the resonator bbox expanded by `preserved_overlap_margin_um` (default **10 µm**).
2. **Associate edge tabs** — if a small polygon boolean-touches a stadium-sized piece in the same window, keep both (shunt resonators often end here with two small tabs).
3. **Augment body interface collar** — when only the stadium shell (~5191 µm²) is found, search body MTE for a separate collar tab that boolean-touches **both** the body stadium and the filter stadium. Add it to preserved MTE (e.g. index 6 → areas **911 + 5191**).

Every series resonator should end with **two** preserved MTE roles: a **stadium** (closed outline) and a **collar** (separate polygon on the mouth). The die-wide interconnect bus (~2096 µm²) may remain in the preserved set for NPI but is **never** chosen as the extension collar in 5.3.

#### `RtegCollectConfig` — tunable parameters

Passed as `config=` to `collect_geometry_roles` (defaults shown):

| Parameter | Default | Role |
|-----------|---------|------|
| `mbe_layer` | `"BAW_MBE"` | Layermap name for ground / preserved MBE |
| `mte_layer` | `"BAW_MTE"` | Layermap name for preserved / body MTE |
| `release_hole_layers` | `("BAW_ReF", "BAW_CAV")` | Layers scanned for release holes |
| `boundary_layer` | `"BAW_EDGE"` | Frame cavity outline layer |
| `preserved_overlap_margin_um` | `10.0` | Bbox expansion when harvesting filter metal |
| `collar_association_gap_um` | `35.0` | Max center-to-center gap to associate a small edge tab with a stadium |
| `stadium_collar_area_um2` | `2500.0` | Area threshold: ≥ this → stadium shell; < this → collar tab |
| `max_edge_collar_area_um2` | `800.0` | Max area for connectMTE tabs associated in the overlap window |
| `min_body_interface_collar_area_um2` | `100.0` | Min area for body-side interface collar augmentation |
| `max_body_interface_collar_area_um2` | `2000.0` | Max area for body-side interface collar augmentation |
| `release_hole_margin_um` | `10.0` | Bbox margin when collecting release holes |
| `filler_bbox_tol_um` | `1.0` | Tolerance for matching filler plates to frame bbox |
| `frame_ring_min_area_um2` | `10000.0` | Min area to treat an MBE ring fragment as the frame ring |
| `min_polygon_area_um2` | `1.0` | Drop degenerate slivers below this area |


In [13]:
from src.rteg_collect import (
    RtegCollectConfig,
    collect_geometry_roles,
    geometry_roles_summary_table,
)

COLLECT_CONFIG = RtegCollectConfig()

all_roles: dict[int, object] = {}
collect_rows: list[dict[str, object]] = []
collect_detail_rows: list[dict[str, object]] = []

for idx in range(len(identification.resonators)):
    res = identification.resonators[idx]
    roles = collect_geometry_roles(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        config=COLLECT_CONFIG,
    )
    all_roles[idx] = roles
    counts = roles.group_counts()
    collect_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            **{k: counts[k] for k in sorted(counts)},
            "total_polygons": sum(counts.values()),
        }
    )
    collect_detail_rows.extend(geometry_roles_summary_table(roles))

collect_overview_df = pd.DataFrame(collect_rows).sort_values("index")
collect_detail_df = pd.DataFrame(collect_detail_rows)

print(f"Collected geometry for {len(all_roles)} resonators\n")
display(collect_overview_df)
print("\nPolygon detail (all resonators):")
display(collect_detail_df)

Collected geometry for 8 resonators



,index,inst_name,res_type,BAW_CAV,BAW_ReF,bottom_ground,center_node,filler_plate,frame_ring,inner_cavity,preserved_mbe,preserved_mte,top_ground,total_polygons
0,0,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,2,2,19
1,1,shuntq3_CDNS_781040784740,shunt,5,1,2,1,1,1,1,3,3,2,20
2,2,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,4,2,2,20
3,3,seriesq3_CDNS_781040784741,series,5,1,2,1,1,1,1,3,3,2,20
4,4,shuntq3_CDNS_781040784742,shunt,7,1,2,1,1,1,1,2,2,2,20
5,5,shuntq3_CDNS_781040784745,shunt,7,1,2,1,1,1,1,1,3,2,20
6,6,seriesq3_CDNS_781040784747,series,5,1,2,1,1,1,1,2,2,2,18
7,7,seriesq3_CDNS_781040784748,series,5,1,2,1,1,1,1,1,2,2,17



Polygon detail (all resonators):


,label,layer,vertices,area_um2,bbox,section,group,index,inst_name
0,pad_top[0],BAW_MBE,4,5408.0,"((120.0, 438.5), (289.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
1,pad_top[1],BAW_MBE,4,3721.0,"((59.0, 409.5), (120.0, 470.5))",ground_plates,top_ground,0,shuntq3_CDNS_781040784740
2,pad_center[0],BAW_MBE,4,3721.0,"((59.0, 259.5), (120.0, 320.5))",ground_plates,center_node,0,shuntq3_CDNS_781040784740
3,pad_bottom[0],BAW_MBE,4,3721.0,"((59.0, 109.5), (120.0, 170.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
4,pad_bottom[1],BAW_MBE,4,5408.0,"((120.0, 109.5), (289.0, 141.5))",ground_plates,bottom_ground,0,shuntq3_CDNS_781040784740
...,...,...,...,...,...,...,...,...,...
149,BAW_CAV[3],BAW_CAV,128,132.7,"((147.0, 293.4), (160.0, 306.4))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
150,BAW_CAV[4],BAW_CAV,8,547.0,"((149.9, 274.5), (182.2, 305.3))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
151,BAW_CAV[5],BAW_CAV,125,11518.4,"((157.9, 223.8), (303.7, 322.6))",release_holes,BAW_CAV,7,seriesq3_CDNS_781040784748
152,inner_cavity,BAW_EDGE,4,196209.0,"((36.5, 36.5), (423.5, 543.5))",frame_boundary,inner_cavity,7,seriesq3_CDNS_781040784748


### 5.2 — Classify GSG nodes by collar orientation

**Files:** `src/rteg_orientation.py`, `src/rteg_classify.py`, `src/rteg_collect.py`

**Entry point:** `classify_nodes(roles, orientation_analysis, config?)`

**What this cell does:** reads each resonator's ground plates and preserved MTE from 5.1, combines with orientation analysis from step 3, and labels which GSG band is signal vs ground. This drives **step 5.4** pad routing; step **5.3 does not use these results**.

#### Routing modes (for 5.4)

| `mte_route_target` | When | Later action |
|--------------------|------|--------------|
| `center_pad` | Preserved MTE faces the center pad | Route extension outer cap to center signal pad |
| `collar_extend` | MTE does not face center | Keep 5.3 extension only; extend toward side pads later |

All MTE geometry must launch from **preserved filter MTE** — never from arbitrary resonator metal.

#### `ClassifyNodesConfig`

Reserved for future tunables — currently empty (no geometry thresholds). Pass `ClassifyNodesConfig()` as shown in the code cell.

#### Overview table columns

| Column | Meaning |
|--------|---------|
| `mte_route_target` | `center_pad` or `collar_extend` |
| `mte_faces_center` | `True` when preserved MTE's nearest GSG band is **center** and (east-west) MTE is on the pad side of the body |
| `signal_terminal` | `MTE` if preserved filter MTE exists; `MBE` if not |
| `signal_drawable` | `True` when preserved MTE exists |
| `facing_pad` | Closest GSG band to the collar (`top` / `center` / `bottom`) |
| `collar_axis` | Collar long axis: `east_west` or `north_south` |
| `top` / `center` / `bottom` | Net assignment per band (`signal` or `ground`) |


In [14]:
from src.rteg_classify import ClassifyNodesConfig, classify_nodes, classification_summary_table
from src.rteg_collect import collect_orientation_inputs

CLASSIFY_CONFIG = ClassifyNodesConfig()

all_classify: dict[int, object] = {}
classify_overview_rows: list[dict[str, object]] = []
classify_detail_rows: list[dict[str, object]] = []

for idx, roles in all_roles.items():
    res = identification.resonators[idx]
    orientation = collect_orientation_inputs(
        rteg_assemblies[idx],
        res,
        identification,
        layermap,
        ground_plates=roles.ground_plates,
        config=COLLECT_CONFIG,
    )
    classification = classify_nodes(
        roles.ground_plates,
        roles.preserved,
        orientation=orientation,
        res_type=res.res_type,
        config=CLASSIFY_CONFIG,
    )
    all_classify[idx] = classification
    collar = classification.collar_orientation
    by_band = classification.by_band()
    classify_overview_rows.append(
        {
            "index": idx,
            "inst_name": roles.inst_name,
            "res_type": res.res_type,
            "method": classification.method,
            "mte_route_target": classification.mte_route_target,
            "mte_faces_center": collar.mte_faces_center,
            "signal_terminal": classification.signal_terminal,
            "signal_drawable": classification.signal_drawable,
            "collar_axis": collar.axis,
            "facing_pad": collar.facing_pad,
            "top": by_band["top"].net,
            "center": by_band["center"].net,
            "bottom": by_band["bottom"].net,
            "note": classification.note,
        }
    )
    classify_detail_rows.extend(
        classification_summary_table(
            classification,
            index=idx,
            inst_name=roles.inst_name,
            res_type=res.res_type,
        )
    )

classify_overview_df = pd.DataFrame(classify_overview_rows).sort_values("index")
classify_df = pd.DataFrame(classify_detail_rows)

print(f"Classified {len(all_classify)} resonators\n")
display(classify_overview_df)

for idx, classification in all_classify.items():
    assert classification.method == "orientation"
    assert classification.mte_route_target in ("center_pad", "collar_extend")
    assert classification.signal_drawable == bool(all_roles[idx].preserved.mte)
    if classification.mte_route_target == "center_pad":
        assert classification.by_band()["center"].net == "signal"

print(f"\nOrientation classification checks passed for all {len(all_classify)} resonators")


Classified 8 resonators



,index,inst_name,res_type,method,mte_route_target,mte_faces_center,signal_terminal,signal_drawable,collar_axis,facing_pad,top,center,bottom,note
0,0,shuntq3_CDNS_781040784740,shunt,orientation,collar_extend,False,MTE,True,east_west,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
1,1,shuntq3_CDNS_781040784740,shunt,orientation,collar_extend,False,MTE,True,east_west,top,ground,ground,ground,preserved MTE not facing center — extend prese...
2,2,seriesq3_CDNS_781040784741,series,orientation,collar_extend,False,MTE,True,east_west,center,ground,ground,ground,preserved MTE not facing center — extend prese...
3,3,seriesq3_CDNS_781040784741,series,orientation,collar_extend,False,MTE,True,east_west,top,ground,ground,ground,preserved MTE not facing center — extend prese...
4,4,shuntq3_CDNS_781040784742,shunt,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — draw MTE span...
5,5,shuntq3_CDNS_781040784745,shunt,orientation,collar_extend,False,MTE,True,north_south,bottom,ground,ground,ground,preserved MTE not facing center — extend prese...
6,6,seriesq3_CDNS_781040784747,series,orientation,center_pad,True,MTE,True,east_west,center,ground,signal,ground,preserved MTE faces center pad — draw MTE span...
7,7,seriesq3_CDNS_781040784748,series,orientation,collar_extend,False,MTE,True,east_west,top,ground,ground,ground,preserved MTE not facing center — extend prese...



Orientation classification checks passed for all 8 resonators


### 5.3 — MTE extensions

**Files:** `src/rteg_mte_extensions.py`

**Entry point:** `build_mte_extensions(all_roles, layermap, config?)`

**What this cell does:** loops over every resonator in `all_roles`, draws one new MTE polygon per resonator, and prints overview + intercept tables. Orientation / 5.2 classification is **not** used in this step.

---

#### How MTE extensions are found — algorithm flow

```
for each resonator index in all_roles:
    collar  ← select_extension_collar(preserved.mte, resonator_body_mte)
    lip     ← find_outward_lip_ab(collar, resonator_body_mte)   # corners A, B
    draw    ← draw_lip_extension(collar, lip, …)                  # rectangle
    DONE?   ← extension_is_connected(draw.polygon, collar, draw)
    store   ← MteExtensionResult(collar, extension, is_connected, …)
```

---

#### 1. How are MTE extensions found?

`build_mte_extensions` is a thin orchestrator. For each resonator it calls `_extension_for_roles`, which chains three geometry functions in order:

| Order | Function | Output |
|-------|----------|--------|
| 1 | `select_extension_collar` | One `TaggedPolygon` — the mouth collar to extend |
| 2 | `find_outward_lip_ab` | `LipIntercept` — corners **A**, **B**, outward normal, lip edge index |
| 3 | `draw_lip_extension` | `CollarExtensionDraw` — new rectangle polygon + merge metadata |

If no preserved MTE exists, the resonator is skipped (no entry in `all_mte`). Otherwise exactly **one** extension polygon is produced.

---

#### 2. For all resonators — make one MTE extension

The code cell passes the full `all_roles` dict (one `RtegGeometryRoles` per index). `build_mte_extensions` returns `dict[int, MteExtensionResult]` with the same keys.

- **Input per resonator:** `roles.preserved.mte` (from 5.1) + `roles.resonator_body_mte`
- **Output per resonator:** `MteExtensionResult` with `collar`, `extension`, `extension_draw`, `is_connected`, `collar_overlap_um2`
- **Golden baseline (KB331):** index **6** — preserved areas `[911, 5191]`, extension on collar **911**, `is_connected == True`

---

#### 3. How is the extension collar found? — `select_extension_collar`

The extension always launches from **preserved filter MTE**, never from arbitrary body metal or a disjoint die-wide bus.

**Candidate filter:** only polygons that are either a **collar tab** (area < `stadium_collar_area_um2`, default 2500 µm²) or a **stadium shell** (area ≥ threshold). Polygons larger than a stadium but not touching (e.g. the ~2096 µm² bus) are excluded.

**Selection priority:**

1. **Associated edge collars** — small tabs that **boolean-touch** a stadium piece in the same preserved set. Prefer the smallest tab that also overlaps `resonator_body_mte` by ≥ `min_collar_overlap_um2`.
2. **Body-overlap fallback** — smallest collar/stadium candidate overlapping the body.
3. **Stadium-only fallback** — if no small tab qualifies, use the stadium shell (tab-mouth search in step 4 handles this case).
4. **Last resort** — smallest preserved MTE piece.

**Series example (index 6):** preserved set = stadium **5191** + collar **911** (touching). Function returns **911** (smaller associated tab), not the stadium shell and not the 2096 bus.

---

#### 4. How is point A found? — `find_outward_lip_ab` (lip edge selection)

Point **A** is not chosen independently. The function scans every edge on the collar polygon and picks one **lip edge** (the extension mouth). **A** and **B** are the two vertices at the ends of that edge.

**Outward direction:** for each candidate edge, compute the outward unit normal — perpendicular to the edge, pointing away from the resonator-body MTE centroid.

**Per-edge score** `(min_merge_um, edge_length_um, body_proximity)`:

- `min_merge_um` — at each corner of the edge, binary-search how far inward (toward the body) a probe can move and still stay inside the collar (`_feasible_merge_um`, target `collar_merge_inset_um`).
- `edge_length_um` — length of the edge (tie-break toward wider mouths).
- `body_proximity` — prefer edges near the collar∩body overlap centroid.

**Stadium collars** (area ≥ 2500 µm²): restrict to **tab-sized mouth edges** between `stadium_tab_mouth_min_um` (12) and `stadium_tab_mouth_max_um` (45). Pick the viable tab with the best score.

**Tab collars** (shunt / series mouth tabs): collect **long edges** — length > max(`lip_long_edge_peak_fraction` × longest edge, `lip_long_edge_min_um`). Prefer edges whose length ≥ `min_mouth_coverage_fraction` × collar bounding-box width. Pick the best-scoring long edge.

The winning edge index is stored in `LipIntercept.lip_edges`; **A** = first vertex, **B** = last vertex on that edge chain.

---

#### 5. How does the lip walk work to find point B?

In the current implementation the lip is a **single collar edge** (not a multi-edge perimeter walk). The "walk" is the **edge scan**: every edge on the collar is evaluated, ranked, and the best outward-facing mouth edge is selected. **B** is simply the other endpoint of that same edge.

Conceptually:

```
for each edge i on collar polygon:
    score[i] ← merge feasibility at both corners + length + body proximity
best_edge ← argmax score (with stadium tab or long-edge filters above)
A ← vertex at start of best_edge
B ← vertex at end of best_edge
outward_normal ← perpendicular to best_edge, away from body
```

The intercept breakdown table in the code cell shows `endcap_index_a`, `collar_intercept_a/b`, and `mouth_span_um` (= distance A→B).

---

#### 6. How is the extension enclosed? — `draw_lip_extension`

Given intercepts A, B and the outward normal **n̂**:

1. **Inward merge** — binary-search `merge_a` and `merge_b` (≤ `collar_merge_inset_um`, default **4 µm**) inward from A and B along **−n̂** so both inner corners stay inside the collar.
2. **Outward extrusion** — outer corners = A and B shifted **+n̂ × `collar_extension_um`** (default **14 µm**).
3. **Touch overlap shift** — shift the whole rectangle slightly further into the collar (`collar_touch_overlap_um`, default **0.5 µm**) to close sub-µm gaps in layout viewers.
4. **Enclose** — build a 4-vertex rectangle: `inner_a → inner_b → outer_b → outer_a` (straight cap between outer corners).

The result is one new `BAW_MTE` polygon tagged on the layermap layer from `mte_layer`.

`_validate_extension` runs immediately after draw: inner corners must be inside the collar; boolean overlap with the collar must exceed `min_collar_overlap_um2`; overlap must not swallow the entire collar (`max_overlap_fraction`).

---

#### 7. DONE state — when is it connected?

`extension_is_connected(ext, collar, draw, cfg)` returns **True** only when the extension is **materially merged** into the collar:

| Check | Threshold | Meaning |
|-------|-----------|---------|
| Boolean overlap area | ≥ `min_collar_overlap_um2` (0.01 µm²) | Extension intersects collar |
| Overlap / extension area | ≥ `min_connection_overlap_fraction` (10%) | Not a hairline sliver |
| Merge inset at A and B | ≥ `min_connection_merge_um` (1 µm) for shunt tabs; ≥ 60% of `min_merge_inset_check_um` for larger collars | Corners pushed inside the collar |
| Mouth coverage (tab collars) | mouth span / collar width ≥ `min_mouth_coverage_fraction` (65%) | Mouth spans the tab |
| Mouth span (stadium collars) | 12–45 µm (`stadium_tab_mouth_min/max_um`) | Tab-sized mouth on stadium shell |

When all checks pass, `MteExtensionResult.is_connected = True`. That is the **DONE** state for step 5.3. The extension is ready for optional pad routing in 5.4.

---

#### Overview table columns (code cell output)

| Column | Meaning |
|--------|---------|
| `n_preserved_mte` | Count of preserved MTE polygons from 5.1 |
| `n_extensions` | Drawn extensions (0 or 1) |
| `is_connected` | Result of `extension_is_connected` (DONE flag) |
| `collar_overlap_um2` | Boolean overlap area between extension and selected collar |

#### Intercept breakdown columns

| Column | Meaning |
|--------|---------|
| `collar_intercept_a` / `collar_intercept_b` | Mouth corners A, B on the collar boundary |
| `endcap_edge_a` / `endcap_edge_b` | Collar edges hosting A and B |
| `merge_inset_a_um` / `merge_inset_b_um` | Achieved inward merge at each corner |
| `mouth_span_um` | Distance A→B (extension mouth width) |
| `extension_um` | Outward extrusion distance (target 14 µm) |

---

#### `MteBuildConfig` — tunable parameters

Edit `mte_cfg` in the code cell below. All fields:

| Parameter | Default | Used by |
|-----------|---------|---------|
| `mte_layer` | `"BAW_MTE"` | Layer assignment for drawn extension |
| `collar_extension_um` | `14.0` | Outward extrusion distance |
| `collar_merge_inset_um` | `4.0` | Target inward merge at A and B |
| `collar_touch_overlap_um` | `0.5` | Extra shift into collar to close viewer gaps |
| `min_collar_overlap_um2` | `0.01` | Min boolean overlap for attachment / connected checks |
| `stadium_collar_area_um2` | `2500.0` | Stadium vs collar tab area split |
| `stadium_edge_area_ratio` | `0.6` | Reserved ratio threshold |
| `lip_long_edge_peak_fraction` | `0.15` | Long-edge filter: ≥ 15% of longest collar edge |
| `lip_long_edge_min_um` | `8.0` | Long-edge absolute minimum length |
| `max_overlap_fraction` | `0.99` | Reject extension that covers nearly all of collar |
| `min_merge_inset_check_um` | `0.5` | Validation floor for merge inset |
| `min_connection_overlap_fraction` | `0.10` | Min overlap/extension area for connected state |
| `min_connection_merge_um` | `1.0` | Min merge inset for shunt tabs |
| `min_mouth_coverage_fraction` | `0.65` | Min mouth/collar-width for tab collars |
| `min_mouth_coverage_shunt_fraction` | `0.85` | Stricter mouth coverage for shunt golden refs 0/1 |
| `collar_association_gap_um` | `35.0` | Gap tolerance (shared semantics with 5.1) |
| `max_edge_collar_area_um2` | `800.0` | Max associated edge tab area |
| `stadium_tab_mouth_min_um` | `12.0` | Min tab mouth on stadium collars |
| `stadium_tab_mouth_max_um` | `45.0` | Max tab mouth on stadium collars |
| `boolean_precision` | `1e-3` | gdstk boolean grid precision (µm) |
| `inside_probe_half_um` | `0.25` | Half-size of point-in-polygon probe |
| `feasible_merge_search_iterations` | `24` | Binary-search iterations for merge depth |


In [15]:
# DRAW MTE extensions for all resonators. 
# Disregard the orientation or position of signal in this step

from src.rteg_mte_extensions import (
    MteBuildConfig,
    build_mte_extensions,
    mte_extensions_overview_rows,
    mte_intercept_breakdown_rows,
)

# --- 5.3 tunables (edit here) ---
mte_cfg = MteBuildConfig(
    mte_layer="BAW_MTE",
    collar_extension_um=14.0,
    collar_merge_inset_um=4.0,
    collar_touch_overlap_um=0.5,
    min_collar_overlap_um2=0.01,
    stadium_collar_area_um2=2500.0,
    stadium_edge_area_ratio=0.6,
    lip_long_edge_peak_fraction=0.15,
    lip_long_edge_min_um=8.0,
    max_overlap_fraction=0.99,
    min_merge_inset_check_um=0.5,
    boolean_precision=1e-3,
    inside_probe_half_um=0.25,
    feasible_merge_search_iterations=24,
)

all_mte = build_mte_extensions(all_roles, layermap, mte_cfg)

inst_names = {idx: roles.inst_name for idx, roles in all_roles.items()}

mte_overview_df = pd.DataFrame(
    mte_extensions_overview_rows(all_mte, inst_names=inst_names)
).sort_values("index")

display(mte_overview_df)
print(f"Drew MTE extensions for {len(all_mte)} resonators")

mte_intercept_df = pd.DataFrame(
    mte_intercept_breakdown_rows(all_mte, inst_names=inst_names)
).sort_values("index")

print("Intercept breakdown (two end-cap edges on preserved collar):")
display(mte_intercept_df)


,index,inst_name,n_preserved_mte,n_extensions,collar_overlap_um2,is_connected
0,0,shuntq3_CDNS_781040784740,2,1,600.38,True
1,1,shuntq3_CDNS_781040784740,3,1,355.52,True
2,2,seriesq3_CDNS_781040784741,2,1,151.94,True
3,3,seriesq3_CDNS_781040784741,3,1,148.44,True
4,4,shuntq3_CDNS_781040784742,2,1,395.13,True
5,5,shuntq3_CDNS_781040784745,3,1,396.67,True
6,6,seriesq3_CDNS_781040784747,2,1,352.59,True
7,7,seriesq3_CDNS_781040784748,2,1,483.98,True


Drew MTE extensions for 8 resonators
Intercept breakdown (two end-cap edges on preserved collar):


,index,inst_name,n_extensions,collar_intercept_a,collar_intercept_b,endcap_edge_a,endcap_edge_b,endcap_index_a,endcap_index_b,lip_edge_index,merge_inset_a_um,merge_inset_b_um,min_merge_um,mouth_span_um,mouth_vertices,extension_um,target_extension_um
0,0,shuntq3_CDNS_781040784740,1,"(268.71, 232.22)","(400.65, 255.62)","(268.71, 232.22) -> (400.65, 255.62)","(268.71, 232.22) -> (400.65, 255.62)",23,23,23,4.5,4.5,4.5,134.00,2,14.0,14.0
1,1,shuntq3_CDNS_781040784740,1,"(297.70, 302.82)","(219.88, 289.22)","(297.70, 302.82) -> (219.88, 289.22)","(297.70, 302.82) -> (219.88, 289.22)",1,1,1,4.5,4.5,4.5,79.00,2,14.0,14.0
2,2,seriesq3_CDNS_781040784741,1,"(241.94, 320.90)","(207.94, 321.30)","(241.94, 320.90) -> (207.94, 321.30)","(241.94, 320.90) -> (207.94, 321.30)",1,1,1,4.5,4.5,4.5,34.00,2,14.0,14.0
3,3,seriesq3_CDNS_781040784741,1,"(152.35, 259.00)","(186.35, 259.00)","(152.35, 259.00) -> (186.35, 259.00)","(152.35, 259.00) -> (186.35, 259.00)",39,39,39,4.5,4.5,4.5,34.00,2,14.0,14.0
4,4,shuntq3_CDNS_781040784742,1,"(142.56, 310.81)","(204.31, 248.41)","(142.56, 310.81) -> (204.31, 248.41)","(142.56, 310.81) -> (204.31, 248.41)",71,71,71,4.5,4.5,4.5,87.79,2,14.0,14.0
5,5,shuntq3_CDNS_781040784745,1,"(224.37, 237.95)","(300.71, 193.88)","(224.37, 237.95) -> (300.71, 193.88)","(224.37, 237.95) -> (300.71, 193.88)",1,1,1,4.5,4.5,4.5,88.15,2,14.0,14.0
6,6,seriesq3_CDNS_781040784747,1,"(158.18, 292.19)","(184.98, 218.56)","(158.18, 292.19) -> (184.98, 218.56)","(158.18, 292.19) -> (184.98, 218.56)",103,103,103,4.5,4.5,4.5,78.35,2,14.0,14.0
7,7,seriesq3_CDNS_781040784748,1,"(280.26, 322.59)","(174.35, 303.91)","(280.26, 322.59) -> (174.35, 303.91)","(280.26, 322.59) -> (174.35, 303.91)",36,36,36,4.5,4.5,4.5,107.55,2,14.0,14.0


### 5.3 export — MTE GDS

**Files:** `src/rteg_mte_extensions.py`, `src/export_gds.py`

**Entry point:** `export_mte_extensions_gds(frame_assemblies, all_mte, output_dir, layermap=…)`

**What this cell does:** writes one GDS per resonator under `draft_output/MTE_generated_deterministic/`. Each file contains the die frame plus the **preserved MTE from 5.1** and the **drawn extension from 5.3** (when `n_extensions > 0`).

| Argument | Role |
|----------|------|
| `frame_assemblies` | Framed resonator cells from step 4 |
| `all_mte` | `build_mte_extensions` output |
| `output_dir` | Destination folder for per-resonator GDS |
| `layermap` | Layer/datatype assignment |
| `parent` | Optional parent cell name in exported hierarchy |
| `flatten` | Default `True` — merge hierarchy for KLayout review |
| `write_lyp` | Default `True` — emit layer properties file |

Re-run this cell after step 5.4 if pad routes were applied (routed net replaces the bare extension in export).


In [16]:
# 5.3 export — writes frame + MTE polygons in ``all_mte`` to GDS.
# Re-run this cell after step 5.4 if pad routes were applied.

from src.export_gds import export_summary_df
from src.rteg_mte_extensions import export_mte_extensions_gds

MTE_OUT = DRAFT / "MTE_generated_deterministic"
mte_export_df = export_summary_df(
    export_mte_extensions_gds(
        rteg_assemblies,
        all_mte,
        MTE_OUT,
        layermap=layermap,
        parent=parent,
    )
)

print(f"Exported {len(mte_export_df)} MTE GDS files to {MTE_OUT}")
display(mte_export_df)

Exported 8 MTE GDS files to C:\Users\santosr4\Documents\rTEG Automation\python_code\draft_output\MTE_generated_deterministic


,index,inst_name,cell_name,path,lyp_path,size_bytes
0,0,shuntq3_CDNS_781040784740,rteg_0_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117406
1,1,shuntq3_CDNS_781040784740,rteg_1_shuntq3_CDNS_781040784740,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,117520
2,2,seriesq3_CDNS_781040784741,rteg_2_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124368
3,3,seriesq3_CDNS_781040784741,rteg_3_seriesq3_CDNS_781040784741,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,124368
4,4,shuntq3_CDNS_781040784742,rteg_4_shuntq3_CDNS_781040784742,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,136034
5,5,shuntq3_CDNS_781040784745,rteg_5_shuntq3_CDNS_781040784745,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,134360
6,6,seriesq3_CDNS_781040784747,rteg_6_seriesq3_CDNS_781040784747,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,122560
7,7,seriesq3_CDNS_781040784748,rteg_7_seriesq3_CDNS_781040784748,C:\Users\santosr4\Documents\rTEG Automation\py...,C:\Users\santosr4\Documents\rTEG Automation\py...,120728


### 5.4 — Route MTE to signal pad

**Files:** `src/rteg_mte_route.py`

**Entry point:** `build_mte_pad_routes(all_mte, all_classify, all_roles, layermap, config?)` *(commented out in notebook — not yet in default run)*

When 5.2 reports `mte_faces_center=True` (`mte_route_target == "center_pad"`), extend the **5.3** outer cap to the **center signal pad** with a heuristic Manhattan/45° corridor. The routed net is boolean-unioned with the collar extension and overlaps the pad by `pad_touch_overlap_um`. **`collar_extend`** resonators keep the 5.3 extension only.

#### `MteRouteConfig` — tunable parameters

| Parameter | Default | Role |
|-----------|---------|------|
| `mte_layer` | `"BAW_MTE"` | Layer for routed net |
| `pad_touch_overlap_um` | `0.5` | Shift route into pad to close viewer gaps |
| `min_pad_overlap_um2` | `0.01` | Min boolean overlap with signal pad |
| `route_width_um` | `None` | Route width; defaults to extension mouth span |
| `min_ground_clearance_um` | `14.0` | DRC clearance from ground MBE |
| `allow_45_degree_corners` | `True` | Permit 45° bends in corridor |
| `boolean_precision` | `1e-3` | gdstk boolean grid precision |
| `inside_probe_half_um` | `0.25` | Point-in-polygon probe half-size |


In [17]:
# from src.export_gds import export_summary_df
# from src.rteg_mte_extensions import export_mte_extensions_gds
# from src.rteg_mte_route import (
#     MteRouteConfig,
#     build_mte_pad_routes,
#     mte_route_overview_rows,
# )

# # --- 5.4 tunables (edit here) ---
# mte_route_cfg = MteRouteConfig(
#     mte_layer="BAW_MTE",
#     pad_touch_overlap_um=0.5,
#     min_pad_overlap_um2=0.01,
#     route_width_um=None,
#     min_ground_clearance_um=14.0,
#     allow_45_degree_corners=True,
#     boolean_precision=1e-3,
#     inside_probe_half_um=0.25,
# )

# all_mte = build_mte_pad_routes(all_roles, all_classify, all_mte, layermap, mte_route_cfg)

# mte_route_df = pd.DataFrame(
#     mte_route_overview_rows(all_mte, all_classify, inst_names=inst_names)
# ).sort_values("index")

# display(mte_route_df)
# routed_count = int(mte_route_df["routed_to_pad"].sum())
# print(f"Routed {routed_count} resonator(s) to center signal pad")

# MTE_OUT = DRAFT / "MTE_generated_deterministic"
# mte_export_df = export_summary_df(
#     export_mte_extensions_gds(
#         rteg_assemblies,
#         all_mte,
#         MTE_OUT,
#         layermap=layermap,
#         parent=parent,
#     )
# )

# print(f"Exported {len(mte_export_df)} MTE GDS files to {MTE_OUT}")